# Run all M3 notebooks on Kaggle in one session

On Kaggle every notebook has its own `/kaggle/working` folder, so notebooks 5 to 13 are run from here,
one after another, in the same session: later notebooks can then use earlier results (13 collects
them all). Each notebook is saved with its outputs in `/kaggle/working/executed/`, and every result in
`/kaggle/working/m3_results.zip`.

1. *File > Import notebook*: this file.
2. *Add input*: your private dataset `veristamp-m3-inputs` (layout in `m3_experiments/README.md`).
3. *Settings*: Accelerator **GPU T4 x2**, Internet **on**.
4. Set the switches below, then *Save Version > Save & Run All (Commit)*. It runs in the background
   (about 2 to 3 hours with inpainting on 200 photos), and the outputs stay with that version.

A failed notebook does not stop the others; the last cell lists which ones failed.

In [ ]:
# Kaggle setup: find the dataset wherever Kaggle put it, copy the code, point the notebooks at the files.
import os, shutil, subprocess, sys
from pathlib import Path

def find(rel, max_depth=6):
    for depth in range(max_depth + 1):
        hits = sorted(Path('/kaggle/input').glob('/'.join(['*'] * depth + [rel])))
        if hits:
            return hits[0]
    return None

if Path('/kaggle/input').is_dir() and not Path('settings.py').exists():
    code = find('m3_experiments/settings.py')
    model = find('m3/best.pt')
    print('Code found at: ', code)
    print('Model found at:', model)
    if code is None or model is None:
        print(subprocess.run(['find', '/kaggle/input', '-maxdepth', '4'], capture_output=True, text=True).stdout[:3000])
        raise SystemExit('The veristamp-m3-inputs dataset is not attached, or has a different layout (see the list above).')
    repo = Path('/kaggle/working/VeriStamp')
    if not (repo / 'm3_experiments' / 'settings.py').exists():
        shutil.rmtree(repo, ignore_errors=True)          # removes a failed earlier clone
        shutil.copytree(code.parent.parent, repo, ignore=shutil.ignore_patterns('.git'))
    # Each item is found on its own, so the inputs can come in several zips or datasets.
    paths = {'M3E_MODEL_PATH': 'm3/best.pt', 'M3E_TEST_DIR': 'm3/test', 'M3E_TEST_MANIFEST': 'm3/test_manifest.csv',
             'M3E_GENIMAGE_SPLIT_DIR': 'genimage/splits', 'M3E_GENIMAGE_ROOT': 'genimage/images',
             'M3E_M1_CHECKPOINT': 'm1/best.pt', 'M3E_M1_VSLIB_DIR': 'm1/vslib_m1_final'}
    for name, rel in paths.items():
        hit = find(rel)
        os.environ[name] = str(hit if hit else model.parent.parent / rel)   # the notebooks run from here inherit these
        print(f'  {rel:<22} {"found" if hit else "not in the inputs"}')
    os.environ['M3E_WORK_DIR'] = '/kaggle/working/m3_work'
    os.chdir(repo / 'm3_experiments')
    for pkg in ('diffusers', 'lpips'):
        try:
            __import__(pkg)
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
print('Working in', Path.cwd())

## Switches

In [ ]:
RUN = ['5_setup_check', '6_rq2_screenshot', '7_image_quality', '8_rq2_vae_roundtrip',
       '9_rq2_inpainting', '10_genimage_check', '11_rq3_detectors', '12_pipeline', '13_report_tables']
# Remove 10, 11 and 12 from RUN until the GenImage split and M1 files are in the dataset.
os.environ['M3E_USE_M1'] = '0'       # '1' once m1/best.pt and m1/vslib_m1_final are in the dataset
os.environ['M3E_USE_M2'] = '0'       # '1' once Supriya's M2 is filled into detectors.py
os.environ['M3E_USE_FUSION'] = '0'   # '1' with both M1 and M2: late fusion (average)

## Run

In [ ]:
import time
out_dir = Path('/kaggle/working/executed')
out_dir.mkdir(parents=True, exist_ok=True)
status = {}
for name in RUN:
    start = time.time()
    r = subprocess.run(['jupyter', 'nbconvert', '--to', 'notebook', '--execute', f'{name}.ipynb',
                        '--ExecutePreprocessor.timeout=-1', '--ExecutePreprocessor.kernel_name=python3',
                        '--output-dir', str(out_dir), '--output', name], capture_output=True, text=True)
    status[name] = 'ok' if r.returncode == 0 else 'FAILED'
    print(f'{name}: {status[name]} ({(time.time() - start) / 60:.1f} min)')
    if r.returncode:
        print(r.stderr[-3000:])

In [ ]:
import shutil
print(shutil.make_archive('/kaggle/working/m3_results', 'zip', 'results'))
print('Executed notebooks with outputs:', out_dir)
failed = [n for n, s in status.items() if s != 'ok']
print('All notebooks ran.' if not failed else f'Failed: {failed} (open them in {out_dir} to see the error)')